# LP3 Machine Learning Practical 3 - Bank Customer Churn using Neural Network

**Problem statement:** Given a bank customer, build a neural network-based classifier that can determine whether they will leave or not in the next 6 months. Dataset: https://www.kaggle.com/barelydedicated/bank-customer-churn-modeling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [ ]:
# Load the dataset (adjust path if needed)
df = pd.read_csv("/content/churn.csv")

# Display basic info
print(df.head())
print(df.info())

In [ ]:
# Drop unnecessary columns
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

# Encode categorical variables
le_gender = LabelEncoder()
df['Gender'] = le_gender.fit_transform(df['Gender'])

# One-hot encode Geography (multiple categories)
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)

# Separate features and target
X = df.drop('Exited', axis=1)
y = df['Exited']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into train-test sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
# Check target distribution
sns.countplot(x='Exited', data=df)
plt.title("Customer Churn Distribution")
plt.show()

In [ ]:
model = Sequential([
    Dense(16, activation='relu', input_dim=X_train.shape[1]),
    Dropout(0.3),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

In [ ]:
# Plot training & validation accuracy/loss
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title('Model Accuracy')

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Model Loss')

plt.show()

In [ ]:
# Predictions
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# Evaluation Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1-Score:", f1)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc = roc_auc_score(y_test, y_pred_prob)

plt.plot(fpr, tpr, label='AUC = %.3f' % roc_auc)
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

## Theory and Explanation
The last cell contains the theory, steps, result interpretation and viva questions as comments.

In [ ]:
# =====================================================================
# THEORY AND EXPLANATION - Practical 3: Bank Customer Churn (Neural Network)
# =====================================================================
#
# 1. PROBLEM
# Given a bank customer's details, predict whether the customer will leave the
# bank (Exited = 1) or stay (Exited = 0). Two classes, so this is BINARY
# CLASSIFICATION, solved here with an Artificial Neural Network (ANN).
#
# 2. KEY THEORY
# a) Artificial Neural Network
#    Layers of connected neurons. Each neuron computes a weighted sum of its
#    inputs plus a bias, then applies an activation function:
#        output = activation( w1*x1 + w2*x2 + ... + b )
#    Input layer takes the features, hidden layers learn patterns, the output layer
#    gives the prediction.
#
# b) Activation functions
#    ReLU    f(z) = max(0, z). Used in hidden layers; fast and avoids vanishing
#            gradients.
#    Sigmoid f(z) = 1 / (1 + e^-z). Squashes the output to 0..1, read as the
#            probability of churn. Used in the output layer for binary problems.
#
# c) Training
#    Forward propagation computes the prediction. The loss function measures the
#    error: binary cross-entropy
#        L = -[ y*log(p) + (1-y)*log(1-p) ]
#    Backpropagation computes the gradient of the loss with respect to every
#    weight, and the optimizer (Adam, an adaptive form of gradient descent) updates
#    the weights. One full pass over the training data is an epoch. The data is fed
#    in small groups called batches (batch_size = 32).
#
# d) Dropout (0.3)
#    During training, 30 percent of the neurons in that layer are randomly switched
#    off in every step. This forces the network not to depend on single neurons and
#    reduces overfitting. It is turned off during prediction.
#
# e) Pre-processing used
#    Drop useless identifier columns (RowNumber, CustomerId, Surname).
#    Label encoding for Gender (two values -> 0/1).
#    One-hot encoding for Geography with drop_first=True (3 countries -> 2 columns).
#    StandardScaler: rescales every feature to mean 0 and standard deviation 1 so
#    that large-valued columns (Balance, Salary) do not dominate the learning.
#    validation_split = 0.2 holds back 20 percent of the training data to monitor
#    overfitting while training.
#
# 3. STEPS (code flow)
#    1. Load churn.csv and inspect it (10000 rows, no missing values).
#    2. Drop ID columns, encode Gender and Geography, split X and y.
#    3. Scale the features and make an 80/20 train-test split.
#    4. Plot the class distribution of Exited (it is imbalanced).
#    5. Build the network and compile it with Adam and binary cross-entropy.
#    6. Train for 50 epochs, batch size 32, with a 20 percent validation split.
#    7. Plot the accuracy and loss curves for training versus validation.
#    8. Predict probabilities on the test set, use a 0.5 threshold to get classes.
#    9. Compute accuracy, precision, recall, F1, the confusion matrix and ROC-AUC.
#
# 4. NETWORK ARCHITECTURE
#    Input (11 features) -> Dense(16, relu) -> Dropout(0.3) -> Dense(8, relu)
#    -> Dense(1, sigmoid)
#    Parameters: 11*16 + 16 = 192, 16*8 + 8 = 136, 8*1 + 1 = 9. Total 337.
#    The 11 features are: CreditScore, Gender, Age, Tenure, Balance, NumOfProducts,
#    HasCrCard, IsActiveMember, EstimatedSalary, Geography_Germany, Geography_Spain.
#
# 5. RESULTS AND HOW TO READ THEM
#    Accuracy 0.863, Precision 0.726, Recall 0.486, F1-score 0.582, AUC 0.853
#    Confusion matrix: TN 1535, FP 72, FN 202, TP 191 (2000 test customers).
#
#    The accuracy looks good but is misleading. About 80 percent of customers do not
#    leave (1607 of 2000 in the test set), so a model that always says "stays" would
#    already reach about 80 percent. The recall of 0.486 shows the model finds only
#    about half of the customers who actually leave (202 missed out of 393). When it
#    does predict churn it is right 73 percent of the time (precision).
#    AUC 0.853 means the model ranks a leaving customer above a staying one about
#    85 percent of the time, so it has useful signal; the weak recall is partly due
#    to the fixed 0.5 threshold and the class imbalance.
#    In the training curves the validation accuracy sits above the training accuracy
#    because dropout is active while computing training accuracy but not validation.
#    The curves flatten near epoch 30 to 40, so there is no strong overfitting.
#    Improvements: class weights or SMOTE for the imbalance, lower the decision
#    threshold (for example 0.3) to raise recall, tune layers and neurons, early
#    stopping.
#    Note: the scaler is fitted on the full data before the split, which lets a
#    little test information leak into training. The cleaner way is to fit the
#    scaler on the training set only.
#
# 6. VIVA QUESTIONS
#    Q: Why sigmoid in the output layer?
#    A: It gives a probability between 0 and 1 for a binary decision.
#    Q: Why binary cross-entropy?
#    A: It is the standard loss for a binary target and penalises confident wrong
#       predictions heavily.
#    Q: What is an epoch, a batch?
#    A: An epoch is one full pass over the training set. A batch is the small group
#       of samples used for one weight update.
#    Q: Why scale the inputs?
#    A: Keeps gradients stable and stops large-valued features dominating.
#    Q: What is dropout?
#    A: Random switching-off of neurons during training to reduce overfitting.
#    Q: Why is accuracy not enough here?
#    A: The data is imbalanced; precision, recall and F1 on the churn class are
#       more meaningful.